# <a id='toc1_'></a>[# §12 Bericht](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [# §12 Bericht](#toc1_)    
  - [📆 data as of](#toc1_1_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [18]:
import os
from pathlib import Path
import duckdb as ddb
import pandas as pd
from connection_helper import sql
from pandas_plots import tbl, pls, hlp
from enum import Enum

hlp.show_package_version()
os.environ['THEME']='dark'
os.environ['DEBUG']='1'

dir_db=Path("C://temp") if hlp.get_os(hlp.OperatingSystem.WINDOWS) else Path(os.path.expanduser("~/tmp"))

file_db_clin = dir_db/'2025-11-11_data_clin.duckdb'
file_db_intern = dir_db/'2025-11-11_internal_clin.duckdb'
file_db_epi = dir_db/'2025-06-20_data_epi.duckdb'
URL_V2 = 'https://gitlab.opencode.de/robert-koch-institut/zentrum-fuer-krebsregisterdaten/cancerdata-references/-/raw/main/data/v2/'

if not file_db_clin.exists():
    raise(FileNotFoundError(f"File {file_db_clin} not found"))

🐍 3.12.8 | 📦 pandas: 2.3.3 | 📦 numpy: 1.26.4 | 📦 duckdb: 1.4.2 | 📦 pandas-plots: 0.23.1 | 📦 connection-helper: 0.13.2


## Datenstand

In [19]:
sql.print_meta(file_db_clin)
# con.close()

database file:           2025-11-11_data_clin.duckdb
data tag:                v2.3
last kkr data import:    2025-09-30
sql table created:       2025-11-11 11:52:01
doi:                     10.18444/5.03.01.0005.0021.0002
document created:        2025-11-28 16:19:03


In [20]:
con = ddb.connect()
_=con.execute("PRAGMA disable_progress_bar;")
_=con.execute(f"ATTACH DATABASE '{file_db_epi}' AS epi (READ_ONLY);")
_=con.execute(f"ATTACH DATABASE '{file_db_intern}' AS internal (READ_ONLY);")
_=con.execute(f"ATTACH DATABASE '{file_db_clin}' AS clin (READ_ONLY); SET SCHEMA 'clin';")

Tumor = con.table("Tumor")
Lieferung = con.table("Lieferung")
dim_icd10_3d = sql.load_file_to_duckdb(con, URL_V2 + "Aggregationen/icd10_3d.csv", sep=";")

In [21]:
# _treecompletion = con.table("internal._treecompletion")
dim_tree = sql.load_file_to_duckdb(con, "../quality-reports/data/dim_tree.csv", sep=";")

_treecompletion = con.sql("select * from internal._treecompletion").project('*, Diagnose_ICD10_Code_DREI as icd10_3d').project('* exclude (Diagnose_ICD10_Code_DREI)')
dim_lieferregister = con.table("dim_lieferregister")

## ⚙️ settings

In [46]:
# # ! settings
class grouper_reg(Enum):
    KKR = "kkr"
    SYSTEM = "system"
    TOTAL = "Total"

class kpi(Enum):
    MISSINGS = "kpi_missing"
    UNKNOWN = "kpi_unknown"
    
class grouper_table(Enum):
    TABLECOL = "tablecolumn"
    TABLES = "tables"

dy_max = 2023

fil_tree_dco_dy=f"dco=0 and dj_clipped>2019 and dj_clipped <= {dy_max}"
fil_tree_c44 = "icd10_3d <> 'C44'"

fil_dy = f"z_dy > 2019 and z_dy <= {dy_max}"
fil_dy_no_c44_no_dco = f"{fil_dy} and not z_is_dco and icd10_3d <> 'C44'"
fil_dy_no_solid_no_dco = f"{fil_dy} and not z_is_dco and is_solid"

## Bericht

### Aktualität

In [23]:
_tu_latest = Tumor.filter(f"z_dy = {dy_max}").aggregate("z_kkr_label, count(*) as cnt")

_tu=(Tumor.set_alias("t")
    .join(Lieferung.set_alias("l"), "l.Lieferregister::int = t.z_kkr::int", "inner")
    .aggregate("t.z_kkr_label, left(cast(max(Diagnosedatum) as varchar),7) as Diagnosemonat, max(Lieferdatum) as Lieferdatum")
    .join(_tu_latest.set_alias("latest"), "z_kkr_label", "left")
    # .project("z_kkr_label, Diagnosemonat")
    .order("z_kkr_label")
    .to_df().set_index("z_kkr_label")
    .astype({"Lieferdatum":str})
    .assign(Tumorfälle_2023 = lambda x: x["cnt"].apply(lambda y: f"{y:_.0f}"))
    .drop(columns=["cnt"])
    .T
)
display(_tu)

z_kkr_label,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH
Diagnosemonat,2023-12,2023-12,2023-12,2023-12,2023-12,2025-04,2023-12,2024-12,2024-12,2023-12,2025-01,2025-01,2025-01,2023-12,2024-11,2023-12
Lieferdatum,2025-01-06,2025-01-29,2025-01-08,2025-02-04,2025-02-23,2025-05-27,2025-02-21,2025-09-30,2025-01-10,2025-02-13,2025-03-10,2025-03-10,2025-01-21,2025-01-13,2025-01-29,2025-02-28
Tumorfälle_2023,36_066,13_937,48_361,5_251,202_729,40_386,28_342,90_457,91_020,10_697,27_295,22_556,23_661,56_030,25_326,15_869


### Vollständigkeit

- es sind folgende Schwellwerte angezeigt:
  - 🟩 0 bis <5%
  - 🟨 5 bis <100%
  - 🟥 bei 100%
- **Filter: `DJ` 2020-2023** Weitere Filter sind extra aufgeführt
- dargestellt sind Variablen aus dem Schema in folgender Notation: `[Elementknoten]Variablenname`
- **Missings**
  - die graumelierte `0` kennzeichnet einen leeren Wert (=keine missings), 0% entsteht durch Rundung von kleinen Werten
- **Unbekannt**
    -  ('U', 'X', 'VX', 'SX', 'okk')
    -  Grading: (U, T)
    -  Morphologie_Code: 8000-8011, 9590, 9591, 9800, 9801, 8050
    -  Lokalisation_Code: C26, C39, C76, C80,  C14.0, C57.9, C63.9, C68.9, C72.9, C75.9
    -  Diagnose_ICD10_Code: ('C80','C80.0', 'C80.1', 'C80.9', 'C79.9')
    -  Diagnosesicherung: 9
    -  Intention, Seite_Zielgebiet, Seitenlokalisation: U
    -  Datum_Genauigkeit: ('M','V') 


In [102]:
# # ? available filter cols: ['kkr', 'table', 'column', 'tablecolumn', 'Diagnose_ICD10_Code_DREI', 'DCO', 'DJ_clipped', <sel criteria>]

from itertools import product


def show_missings(kpi_type: str, filter: str = "", group_table: str = grouper_table.TABLECOL.value):

    if filter and group_table == grouper_table.TABLES.value:
        print("❌ no filter allowed on tables agg level")
        return

    if not filter:
        filter="True"

    # * start
    db_tree =_treecompletion
    
    # * only join if columm is involved
    if group_table == grouper_table.TABLECOL.value:
        db_tree= (db_tree
                .set_alias("t2")
                .join(dim_tree.set_alias("tr"), f"t2.{group_table}=tr.{group_table}", "inner")
                .project("* exclude (t2.tables, t2.tablecolumn)")
        )


    # * then go on, create df
    df_tree =(db_tree.set_alias("t")
        .join(dim_lieferregister.set_alias("l"), "t.kkr::int=l.code::int", "inner")
        .join(dim_icd10_3d.set_alias("i"), "icd10_3d = i.code", "left")
        # * we want the enriched kkr str
        .project("* replace (l.kkr as kkr)")
        # * to use filter, dim_tree must have been joint
        .filter(filter)
        .aggregate(f"""--sql
                kkr,
                {group_table}, --, Diagnose_ICD10_Code_DREI, DCO, DJ_clipped
                sum(allHasValues) as allHasValues,
                sum(sumIsMissing) as sumIsMissing,
                sum(sumIsUnknown) as sumIsUnknown,
                sum(sumRelatedTumors) as sumRelatedTumors,
        """)
        .project(f"""--sql
                kkr,
                {group_table}, --, Diagnose_ICD10_Code_DREI, DCO, DJ_clipped
                case
                    when allHasValues = 0 then 1
                    else case
                        when sumRelatedTumors = 0 then 0
                        else sumIsMissing / sumRelatedTumors
                    end
                end as kpi_missing,
                case
                    when sumRelatedTumors = 0 then 0
                    else sumIsUnknown / sumRelatedTumors
                end as kpi_unknown,
            """
            )
        .to_df()
        # .pivot(index=group_table, columns="kkr", values=kpi_type)
    )

    # * create the table for Totals
    df_tree_total =(db_tree.set_alias("t")
        .join(dim_icd10_3d.set_alias("i"), "icd10_3d = i.code", "left")
        # * to use filter, dim_tree must have been joint
        .filter(filter)
        .aggregate(f"""--sql
                {group_table}, --, Diagnose_ICD10_Code_DREI, DCO, DJ_clipped
                sum(allHasValues) as allHasValues, sum(sumIsMissing) as sumIsMissing, sum(sumIsUnknown) as sumIsUnknown, sum(sumRelatedTumors) as sumRelatedTumors
                """)
        .project(f"""--sql
                'Total' as kkr, {group_table} --, Diagnose_ICD10_Code_DREI, DCO, DJ_clipped
                ,case when allHasValues = 0 then 1 else case when sumRelatedTumors = 0 then 0 else sumIsMissing / sumRelatedTumors end end as kpi_missing
                ,case when sumRelatedTumors = 0 then 0 else sumIsUnknown / sumRelatedTumors end as kpi_unknown
            """
            )
        .to_df()
        # .pivot(index=group_table, columns="kkr", values=kpi_type)
    )
    # display(df_tree_total)

    # * concat both tables
    # df_all = df_tree
    # ! dt_tree_total is still experimental
    df_all = pd.concat([df_tree, df_tree_total], axis=0)

    # * debug
    # print(df_all["tablecolumn"].unique())
    # display(df_all)

    # * add missing kkr
    list_kkr_all = dim_lieferregister.filter("kkr <> 'Total'").to_df().kkr.unique().tolist()
    list_kkr_present = df_all.kkr.unique().tolist()
    list_kkr = list(set(list_kkr_all) - set(list_kkr_present))
    
    # * get available columns 
    list_col = df_all.tablecolumn.unique().tolist()
    
    # * multiply
    prod = list(product(list_kkr, list_col))
    
    # * add this frame to also have missxing kkr
    df_add = pd.DataFrame(prod, columns=["kkr","tablecolumn"]).assign(kpi_missing=1).assign(kpi_unknown=0)
    df_all = pd.concat([df_all,df_add], axis=0).reset_index(drop=True)

    df_all= (df_all
        .pivot(index=group_table, columns="kkr", values=kpi_type)
        # ! when no kkr data are present, set to 100% missing
        .fillna(1)
    )
    
    out = tbl.show_num_df(df_all,
            data_bar_axis="xy",
            total_axis="",
            pct_axis="",
            precision=1,
            kpi_mode="rag_abs",
            # * red threshold: 100% missings | 35% unknown
            kpi_rag_list=[0.05, 1] if kpi_type == kpi.MISSINGS.value else [0.05, 0.35],
            show_as_pct=True,
        )
    display(out)

#### Personenangaben

In [103]:
FILTER_PAT = """--sql
    tables in ('Patient')
"""

In [104]:
show_missings(
    kpi.MISSINGS.value,
    filter=FILTER_PAT,
    group_table=grouper_table.TABLECOL.value,
)

kkr,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
tablecolumn,,,,,,,,,,,,,,,,,
[Patient]Datum_Vitalstatus,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Patient]Datum_Vitalstatus_Genauigkeit,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Patient]Geburtsdatum,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Patient]Geburtsdatum_Genauigkeit,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Patient]Geschlecht,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Patient]Verstorben,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩


In [105]:
show_missings(
    kpi.UNKNOWN.value,
    # filter=f"sel5_pflicht",
    filter="tables in ('Patient')",
    group_table=grouper_table.TABLECOL.value,
)

kkr,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
tablecolumn,,,,,,,,,,,,,,,,,
[Patient]Datum_Vitalstatus,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Patient]Datum_Vitalstatus_Genauigkeit,0 🟩,0 🟩,0.0% 🟩,0 🟩,0.1% 🟩,0.4% 🟩,0 🟩,0.0% 🟩,0.9% 🟩,0.0% 🟩,0.3% 🟩,0.6% 🟩,2.2% 🟩,0.1% 🟩,0.7% 🟩,4.4% 🟩,0.4% 🟩
[Patient]Geburtsdatum,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Patient]Geburtsdatum_Genauigkeit,0 🟩,0 🟩,0 🟩,0 🟩,0.2% 🟩,0.2% 🟩,0.0% 🟩,0.0% 🟩,0.0% 🟩,0 🟩,0.0% 🟩,0.0% 🟩,0.1% 🟩,0.0% 🟩,0.0% 🟩,0 🟩,0.1% 🟩
[Patient]Geschlecht,0.0% 🟩,0.0% 🟩,0.0% 🟩,0.0% 🟩,0.0% 🟩,0 🟩,0.0% 🟩,0.0% 🟩,0.0% 🟩,0.0% 🟩,0.0% 🟩,0 🟩,0.0% 🟩,0 🟩,0 🟩,0.0% 🟩,0.0% 🟩
[Patient]Verstorben,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩


#### Tumorangaben

In [106]:
FILTER_TUM = f"""--sql
        {fil_tree_dco_dy} and {fil_tree_c44} and
        tablecolumn in (
            '[Tumor]Diagnosesicherung',
            '[Tumor]T_p',
            '[Tumor]UICC_Stadium_p',
            '[Tumor]Morphologie_Code',
            '[Tumor]Topographie_Code',
            '[Tumor]Grading',
            )
"""

In [107]:
show_missings(
    kpi.MISSINGS.value,
    filter=FILTER_TUM,
    group_table=grouper_table.TABLECOL.value,
)

kkr,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
tablecolumn,,,,,,,,,,,,,,,,,
[Tumor]Diagnosesicherung,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Tumor]Grading,0 🟩,0 🟩,4.8% 🟩,0 🟩,0 🟩,1.3% 🟩,8.7% 🟨,0 🟩,2.4% 🟩,0 🟩,0.1% 🟩,0.2% 🟩,0.0% 🟩,0.0% 🟩,0.3% 🟩,0.6% 🟩,1.3% 🟩
[Tumor]Morphologie_Code,0 🟩,0 🟩,4.8% 🟩,0 🟩,0 🟩,1.3% 🟩,8.7% 🟨,0 🟩,2.4% 🟩,0 🟩,0.1% 🟩,0.2% 🟩,0.0% 🟩,0.0% 🟩,0.3% 🟩,0.6% 🟩,1.3% 🟩
[Tumor]T_p,39.3% 🟨,51.0% 🟨,46.0% 🟨,45.2% 🟨,50.1% 🟨,48.1% 🟨,51.3% 🟨,55.4% 🟨,44.8% 🟨,44.0% 🟨,52.8% 🟨,51.6% 🟨,46.4% 🟨,44.8% 🟨,52.0% 🟨,46.3% 🟨,48.8% 🟨
[Tumor]Topographie_Code,0 🟩,0.1% 🟩,2.7% 🟩,0 🟩,0 🟩,0 🟩,2.8% 🟩,1.4% 🟩,0.3% 🟩,0 🟩,0.0% 🟩,0.0% 🟩,0.2% 🟩,0.0% 🟩,0.0% 🟩,0.2% 🟩,0.6% 🟩
[Tumor]UICC_Stadium_p,51.6% 🟨,100.0% 🟥,100.0% 🟨,54.8% 🟨,100.0% 🟥,64.9% 🟨,74.2% 🟨,79.3% 🟨,67.7% 🟨,92.2% 🟨,63.0% 🟨,62.8% 🟨,57.6% 🟨,47.0% 🟨,54.8% 🟨,51.9% 🟨,77.3% 🟨


In [108]:
show_missings(
    kpi.UNKNOWN.value,
    filter=FILTER_TUM,
    group_table=grouper_table.TABLECOL.value,
)

kkr,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
tablecolumn,,,,,,,,,,,,,,,,,
[Tumor]Diagnosesicherung,5.4% 🟨,6.3% 🟨,8.2% 🟨,18.9% 🟨,6.7% 🟨,2.0% 🟩,11.1% 🟨,9.7% 🟨,1.2% 🟩,24.8% 🟨,1.0% 🟩,0.7% 🟩,0.6% 🟩,0.1% 🟩,1.0% 🟩,0.7% 🟩,5.4% 🟨
[Tumor]Grading,40.3% 🟥,55.6% 🟥,40.8% 🟥,36.0% 🟥,44.2% 🟥,46.8% 🟥,88.0% 🟥,50.3% 🟥,45.2% 🟥,36.3% 🟥,45.1% 🟥,45.5% 🟥,43.6% 🟥,49.9% 🟥,35.5% 🟥,31.9% 🟨,47.0% 🟥
[Tumor]Morphologie_Code,4.7% 🟩,6.1% 🟨,1.4% 🟩,3.9% 🟩,7.3% 🟨,1.6% 🟩,1.9% 🟩,9.8% 🟨,1.0% 🟩,4.0% 🟩,3.9% 🟩,2.8% 🟩,2.4% 🟩,3.3% 🟩,3.4% 🟩,2.4% 🟩,4.5% 🟩
[Tumor]T_p,5.4% 🟨,1.0% 🟩,0.2% 🟩,0.3% 🟩,1.1% 🟩,0.2% 🟩,0.6% 🟩,0.3% 🟩,0.1% 🟩,0.7% 🟩,0.3% 🟩,0.2% 🟩,0.6% 🟩,0.1% 🟩,0.2% 🟩,0.2% 🟩,0.7% 🟩
[Tumor]Topographie_Code,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Tumor]UICC_Stadium_p,0.0% 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0.0% 🟩,0 🟩,0 🟩,0.0% 🟩,0 🟩,0.0% 🟩,0 🟩,0 🟩,0.0% 🟩,0 🟩,0 🟩,0.0% 🟩


#### Organmodule - Mamma

In [109]:
FILTER_MAMMA = f"""--sql
        {fil_tree_dco_dy} and icd10_3d = 'C50' and
        tablecolumn in (
            '[Tumor]Praetherapeutischer_Menopausenstatus',
            '[Tumor]HormonrezeptorStatus_Oestrogen',
            '[Tumor]HormonrezeptorStatus_Progesteron',
            '[Tumor]Her2neuStatus',

            '[Tumor]RASMutation',
            '[Tumor]ScoreErgebnis',
            '[Tumor]DatumPSA',
            '[Tumor]DatumPSA_Genauigkeit',
            )
"""

show_missings(
    kpi.MISSINGS.value,
    filter=FILTER_MAMMA,
    group_table=grouper_table.TABLECOL.value,
)

show_missings(
    kpi.UNKNOWN.value,
    filter=FILTER_MAMMA,
    group_table=grouper_table.TABLECOL.value,
)

kkr,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
tablecolumn,,,,,,,,,,,,,,,,,
[Tumor]DatumPSA,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥
[Tumor]DatumPSA_Genauigkeit,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥
[Tumor]Her2neuStatus,16.9% 🟨,100.0% 🟥,6.2% 🟨,1.0% 🟩,40.9% 🟨,18.0% 🟨,35.7% 🟨,11.9% 🟨,13.1% 🟨,5.7% 🟨,0.6% 🟩,0.9% 🟩,4.0% 🟩,0.9% 🟩,8.5% 🟨,9.4% 🟨,19.7% 🟨
[Tumor]HormonrezeptorStatus_Oestrogen,14.3% 🟨,100.0% 🟥,6.2% 🟨,1.0% 🟩,40.1% 🟨,17.6% 🟨,35.8% 🟨,11.8% 🟨,29.5% 🟨,5.4% 🟨,7.9% 🟨,8.2% 🟨,18.0% 🟨,2.9% 🟩,11.2% 🟨,5.6% 🟨,22.7% 🟨
[Tumor]HormonrezeptorStatus_Progesteron,14.4% 🟨,100.0% 🟥,6.2% 🟨,1.0% 🟩,40.0% 🟨,17.3% 🟨,35.8% 🟨,16.1% 🟨,29.5% 🟨,5.2% 🟨,8.0% 🟨,8.3% 🟨,18.0% 🟨,2.9% 🟩,11.3% 🟨,5.6% 🟨,23.3% 🟨
[Tumor]Praetherapeutischer_Menopausenstatus,21.1% 🟨,100.0% 🟥,39.0% 🟨,6.3% 🟨,34.4% 🟨,39.5% 🟨,43.2% 🟨,29.7% 🟨,43.0% 🟨,27.1% 🟨,24.9% 🟨,34.3% 🟨,24.0% 🟨,9.3% 🟨,34.5% 🟨,42.4% 🟨,35.2% 🟨
[Tumor]RASMutation,100.0% 🟥,100.0% 🟥,100.0% 🟨,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟨
[Tumor]ScoreErgebnis,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥


kkr,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
tablecolumn,,,,,,,,,,,,,,,,,
[Tumor]DatumPSA,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Tumor]DatumPSA_Genauigkeit,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Tumor]Her2neuStatus,2.0% 🟩,0 🟩,1.9% 🟩,2.5% 🟩,3.9% 🟩,8.4% 🟨,5.0% 🟨,1.9% 🟩,10.3% 🟨,2.1% 🟩,8.5% 🟨,6.5% 🟨,11.8% 🟨,6.6% 🟨,14.3% 🟨,2.7% 🟩,5.3% 🟨
[Tumor]HormonrezeptorStatus_Oestrogen,0.8% 🟩,0 🟩,0.8% 🟩,1.0% 🟩,1.9% 🟩,1.6% 🟩,2.1% 🟩,0.4% 🟩,0.1% 🟩,0.5% 🟩,2.1% 🟩,0.3% 🟩,0.4% 🟩,3.7% 🟩,0.6% 🟩,0.4% 🟩,1.1% 🟩
[Tumor]HormonrezeptorStatus_Progesteron,0.7% 🟩,0 🟩,0.9% 🟩,1.0% 🟩,2.0% 🟩,2.3% 🟩,2.1% 🟩,0.4% 🟩,0.1% 🟩,0.8% 🟩,2.1% 🟩,0.3% 🟩,0.5% 🟩,3.6% 🟩,0.6% 🟩,0.4% 🟩,1.2% 🟩
[Tumor]Praetherapeutischer_Menopausenstatus,6.7% 🟨,0 🟩,1.5% 🟩,1.1% 🟩,5.8% 🟨,1.3% 🟩,1.6% 🟩,1.0% 🟩,2.5% 🟩,8.1% 🟨,5.2% 🟨,4.9% 🟩,2.3% 🟩,3.6% 🟩,5.0% 🟨,0.5% 🟩,3.2% 🟩
[Tumor]RASMutation,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Tumor]ScoreErgebnis,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩


##### Organmodule - Prostata

In [110]:
FILTER_PROSTATA = f"""--sql
        {fil_tree_dco_dy} and icd10_3d = 'C61' and
        tablecolumn in (
            '[Tumor]ScoreErgebnis',
            '[Tumor]AnlassGleasonScore',
            '[Tumor]DatumPSA',
            '[Tumor]DatumPSA_Genauigkeit',
            )
"""

show_missings(
    kpi.MISSINGS.value,
    filter=FILTER_PROSTATA,
    group_table=grouper_table.TABLECOL.value,
)

show_missings(
    kpi.UNKNOWN.value,
    filter=FILTER_PROSTATA,
    group_table=grouper_table.TABLECOL.value,
)

kkr,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
tablecolumn,,,,,,,,,,,,,,,,,
[Tumor]AnlassGleasonScore,20.5% 🟨,100.0% 🟥,10.4% 🟨,7.7% 🟨,100.0% 🟥,65.3% 🟨,56.4% 🟨,13.9% 🟨,54.7% 🟨,8.2% 🟨,46.0% 🟨,56.4% 🟨,40.8% 🟨,39.2% 🟨,74.7% 🟨,87.8% 🟨,53.2% 🟨
[Tumor]DatumPSA,42.2% 🟨,100.0% 🟥,39.6% 🟨,36.5% 🟨,100.0% 🟥,45.6% 🟨,63.9% 🟨,95.5% 🟨,44.0% 🟨,27.4% 🟨,23.1% 🟨,21.5% 🟨,19.9% 🟨,13.3% 🟨,28.9% 🟨,17.8% 🟨,59.7% 🟨
[Tumor]DatumPSA_Genauigkeit,42.2% 🟨,100.0% 🟥,39.6% 🟨,36.5% 🟨,100.0% 🟥,45.6% 🟨,63.9% 🟨,95.5% 🟨,44.0% 🟨,27.4% 🟨,23.1% 🟨,21.5% 🟨,19.9% 🟨,13.3% 🟨,28.9% 🟨,17.8% 🟨,59.7% 🟨
[Tumor]ScoreErgebnis,18.5% 🟨,100.0% 🟥,8.5% 🟨,7.7% 🟨,100.0% 🟥,20.7% 🟨,52.6% 🟨,10.5% 🟨,22.0% 🟨,4.1% 🟩,9.5% 🟨,8.6% 🟨,12.3% 🟨,4.2% 🟩,8.1% 🟨,9.4% 🟨,35.1% 🟨


kkr,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
tablecolumn,,,,,,,,,,,,,,,,,
[Tumor]AnlassGleasonScore,1.7% 🟩,0 🟩,0.7% 🟩,0.5% 🟩,0 🟩,0 🟩,0.6% 🟩,0.5% 🟩,3.0% 🟩,0.5% 🟩,4.0% 🟩,1.7% 🟩,0.1% 🟩,1.1% 🟩,0.5% 🟩,0.0% 🟩,1.0% 🟩
[Tumor]DatumPSA,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Tumor]DatumPSA_Genauigkeit,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0.0% 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0.0% 🟩
[Tumor]ScoreErgebnis,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩


##### Organmodule - Melanom

In [111]:
FILTER_MELANOM = f"""--sql
        {fil_tree_dco_dy} and icd10_3d = 'C43' and
        tablecolumn in (
            '[Tumor]Tumordicke',
            '[Tumor]LDH',
            '[Tumor]Ulzeration',
            )
"""

show_missings(
    kpi.MISSINGS.value,
    filter=FILTER_MELANOM,
    group_table=grouper_table.TABLECOL.value,
)

show_missings(
    kpi.UNKNOWN.value,
    filter=FILTER_MELANOM,
    group_table=grouper_table.TABLECOL.value,
)

kkr,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
tablecolumn,,,,,,,,,,,,,,,,,
[Tumor]LDH,98.4% 🟨,100.0% 🟥,96.3% 🟨,87.0% 🟨,100.0% 🟥,100.0% 🟥,97.7% 🟨,91.9% 🟨,100.0% 🟥,97.3% 🟨,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,98.0% 🟨
[Tumor]Tumordicke,56.5% 🟨,100.0% 🟥,47.3% 🟨,21.2% 🟨,100.0% 🟥,100.0% 🟥,78.5% 🟨,17.1% 🟨,100.0% 🟥,46.0% 🟨,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,77.2% 🟨
[Tumor]Ulzeration,62.7% 🟨,100.0% 🟥,64.2% 🟨,16.9% 🟨,100.0% 🟥,100.0% 🟥,82.3% 🟨,42.2% 🟨,100.0% 🟥,61.5% 🟨,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,100.0% 🟥,83.6% 🟨


kkr,01-SH,02-HH,03-NI,04-HB,05-NW,06-HE,07-RP,08-BW,09-BY,10-SL,11-BE,12-BB,13-MV,14-SN,15-ST,16-TH,Total
tablecolumn,,,,,,,,,,,,,,,,,
[Tumor]LDH,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Tumor]Tumordicke,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
[Tumor]Ulzeration,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩,0 🟩
